In [ ]:
import random
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_absolute_experiment
from vpei.epistemic_consistency.experiment_utils import print_absolute_experiment_results

cvs_df = pd.read_csv('./data/cvs.csv')
jd_df = pd.read_csv('./data/job_descriptions.csv')

cvs_sorted = cvs_df.sort_values('profession').reset_index(drop=True)
jd_sorted = jd_df.sort_values('profession').reset_index(drop=True)

df = pd.DataFrame({
    'profession': cvs_sorted['profession'],
    'cv': cvs_sorted['cv'],
    'job_description': jd_sorted['job_description'],
})
df

In [ ]:
experiment_name = "cvs"
system_prompt = EXPERIMENTS[experiment_name]["absolute_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["absolute_experiment"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
name = "J.S."
row = df.iloc[0]
user_prompt = user_prompt_template.format(
    name=name,
    political_attitude="Republican",
    job_description=row['job_description'],
    cv=row['cv']
)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
response = make_llm_request(model_name, messages, **model_kwargs)
print("Response:", response)

In [ ]:
models = ["gpt-5-mini"]

n = 20
stimuli_factors = ["job_description", "cv"]
additional_variables_from_df_to_save = ["profession"]
custom_model_kwargs = {}
random_seed = 42
path_to_save_model_outputs = "./absolute_experiment/"

In [ ]:
payloads = await carry_out_absolute_experiment(
    models=models,
    df=df,
    n=n,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
    stimuli_factors=stimuli_factors,
    additional_variables_from_df_to_save=additional_variables_from_df_to_save,
    custom_model_kwargs=custom_model_kwargs,
    path_to_save_model_outputs=path_to_save_model_outputs,
    random_seed=random_seed
)

print_absolute_experiment_results(payloads, models)